# Ingestion Code test and Data Quality Check

In [0]:
#Api Connection Check

import requests

url = "https://hapi.fhir.org/baseR4/Patient?_count=5"

response = requests.get(url, timeout=30)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))

data = response.json()

print("Resource type:", data.get("resourceType"))
print("Number of records:", len(data.get("entry", [])))

In [0]:
from datetime import datetime, timedelta

# FHIR API configuration check
BASE_URL = "https://hapi.fhir.org/baseR4"

# Resources required by the assignment
FHIR_RESOURCES = [
    "Patient",
    "Encounter",
    "Observation",
    "Condition"
]

# Number of records per API page
PAGE_SIZE = 100

# Ingestion window: last 3 days
END_DATE = datetime.utcnow().date()
START_DATE = END_DATE - timedelta(days=2)

print("Start date:", START_DATE)
print("End date:", END_DATE)
print("Resources:", FHIR_RESOURCES)

In [0]:
# Pagination base function check
import requests
import time

def fetch_fhir_resource(resource, page_size=100):
  

    url = f"{BASE_URL}/{resource}?_count={page_size}"
    
    all_records = []
    page_number = 1

    while url:
        print(f"Fetching {resource} - page {page_number}")

        response = requests.get(url, timeout=60)
        response.raise_for_status()

        bundle = response.json()

        entries = bundle.get("entry", [])

        for entry in entries:
            resource_data = entry.get("resource")
            if resource_data:
                all_records.append(resource_data)

        
        next_url = None

        for link in bundle.get("link", []):
            if link.get("relation") == "next":
                next_url = link.get("url")
                break

        url = next_url
        page_number += 1

        
        time.sleep(0.2)

    print(f"{resource}: {len(all_records)} records fetched")

    return all_records

In [0]:
patients = fetch_fhir_resource("Patient", PAGE_SIZE)

print("Total patients:", len(patients))

In [0]:
#Function_test

import requests
import json
import os
import time
from datetime import datetime, timezone


RAW_BASE_PATH = "/Volumes/fhir_assignment/raw/fhir_files"


def ingest_fhir_resource(resource, start_date, end_date, page_size=100):
   

    url = (
        f"{BASE_URL}/{resource}"
        f"?_lastUpdated=ge{start_date}T00:00:00"
        f"&_lastUpdated=lt{end_date}T00:00:00"
        f"&_count={page_size}"
    )

    extraction_timestamp = datetime.now(timezone.utc).isoformat()

    page_number = 1
    total_records = 0

    raw_path = (
        f"{RAW_BASE_PATH}/"
        f"{resource}/"
        f"{start_date}_{end_date}"
    )

    os.makedirs(raw_path, exist_ok=True)

    while url:

        api_call_timestamp = datetime.now(timezone.utc).isoformat()

        print(f"Fetching {resource} - page {page_number}")

        response = requests.get(url, timeout=60)
        response.raise_for_status()

        bundle = response.json()

        records_in_page = len(bundle.get("entry", []))
        total_records += records_in_page

        metadata = {
            "resource": resource,
            "page_number": page_number,
            "extraction_timestamp": extraction_timestamp,
            "api_call_timestamp": api_call_timestamp,
            "api_url_or_params": url,
            "records_in_page": records_in_page,
            "data_saved_timestamp": datetime.now(timezone.utc).isoformat()
        }


        file_name = f"page_{page_number}.json"
        file_path = f"{raw_path}/{file_name}"

        with open(file_path, "w") as f:
            json.dump(bundle, f)

        metadata_file = f"{raw_path}/page_{page_number}_metadata.json"

        with open(metadata_file, "w") as f:
            json.dump(metadata, f, indent=2)

        print(f"  Records in page: {records_in_page}")

        next_url = None

        for link in bundle.get("link", []):
            if link.get("relation") == "next":
                next_url = link.get("url")
                break

        url = next_url
        page_number += 1

        time.sleep(0.2)

    print(f"\n{resource} ingestion complete")
    print(f"Total records: {total_records}")
    print(f"Raw location: {raw_path}")

    return {
        "resource": resource,
        "total_records": total_records,
        "extraction_timestamp": extraction_timestamp,
        "raw_path": raw_path
    }

In [0]:
patient_result = ingest_fhir_resource(
    resource="Patient",
    start_date=START_DATE,
    end_date=END_DATE,
    page_size=PAGE_SIZE
)

print(patient_result)

In [0]:
patients[0]

# Bronze code test and Data Quality Check

In [0]:
# first df
RAW_PATIENT_PATH = (
    "/Volumes/fhir_assignment/raw/fhir_files/"
    "Patient/2026-08-14_2026-08-16"
)

patient_raw_df = spark.read.json(
    f"{RAW_PATIENT_PATH}/page_*.json"
)

display(patient_raw_df)

In [0]:
from pyspark.sql.functions import explode, col

patient_entries_df = (
    patient_raw_df
    .select(explode(col("entry")).alias("entry"))
)

display(patient_entries_df)

In [0]:
from pyspark.sql.functions import col, from_json

entry_schema = (
    spark.read
    .json(
        patient_entries_df
        .select("entry")
        .rdd
        .map(lambda row: row["entry"])
    )
    .schema
)

patient_parsed_df = patient_entries_df.withColumn(
    "entry_parsed",
    from_json(col("entry"), entry_schema)
)

display(patient_parsed_df)

In [0]:
from pyspark.sql.functions import col, explode, get_json_object

patient_bronze_test = (
    patient_raw_df
    .select(explode(col("entry")).alias("entry"))
    .select(
        get_json_object(col("entry"), "$.resource.resourceType").alias("resource_type"),
        get_json_object(col("entry"), "$.resource.id").alias("resource_id"),
        get_json_object(col("entry"), "$.resource").alias("resource_json")
    )
)

display(patient_bronze_test)

In [0]:
from pyspark.sql.functions import col, explode, to_json

patient_bronze_test = (
    patient_raw_df
    .select(explode(col("entry")).alias("entry"))
    .select(
        col("entry.resource.resourceType").alias("resource_type"),
        col("entry.resource.id").alias("resource_id"),
        to_json(col("entry.resource")).alias("resource_json")
    )
)

display(patient_bronze_test)

In [0]:
raw_count = patient_entries_df.count()
bronze_count = patient_bronze_test.count()

print("Raw entry count   :", raw_count)
print("Bronze record count:", bronze_count)

assert raw_count == bronze_count, (
    f"Count mismatch: raw={raw_count}, bronze={bronze_count}"
)

print("PASS: Raw and Bronze record counts match")

In [0]:
null_id_count = patient_bronze_test.filter(
    col("resource_id").isNull()
).count()

print("Null resource_id count:", null_id_count)

assert null_id_count == 0, (
    f"Found {null_id_count} records with null resource_id"
)

print("PASS: No null resource_id values")

In [0]:
duplicate_id_count = (
    patient_bronze_test
    .groupBy("resource_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate resource_id count:", duplicate_id_count)

assert duplicate_id_count == 0, (
    f"Found {duplicate_id_count} duplicate resource_id values"
)

print("PASS: No duplicate resource_id values")

In [0]:
invalid_resource_type_count = (
    patient_bronze_test
    .filter(col("resource_type") != "Patient")
    .count()
)

print("Invalid resource_type count:", invalid_resource_type_count)

assert invalid_resource_type_count == 0, (
    f"Found {invalid_resource_type_count} records with invalid resource_type"
)

print("PASS: All records are Patient resources")

In [0]:
null_json_count = (
    patient_bronze_test
    .filter(
        col("resource_json").isNull() |
        (col("resource_json") == "")
    )
    .count()
)

print("Null/empty resource_json count:", null_json_count)

assert null_json_count == 0, (
    f"Found {null_json_count} records with null/empty resource_json"
)

print("PASS: All records contain resource_json")

In [0]:
from pyspark.sql.functions import input_file_name

RAW_BASE_PATH = "/Volumes/fhir_assignment/raw/fhir_files"

patient_raw_df = (
    spark.read
    .option("multiLine", True)
    .json(f"{RAW_BASE_PATH}/Patient/2026-08-14_2026-08-16/*.json")
)

encounter_raw_df = (
    spark.read
    .option("multiLine", True)
    .json(f"{RAW_BASE_PATH}/Encounter/2026-08-14_2026-08-16/*.json")
)

observation_raw_df = (
    spark.read
    .option("multiLine", True)
    .json(f"{RAW_BASE_PATH}/Observation/2026-08-14_2026-08-16/*.json")
)

condition_raw_df = (
    spark.read
    .option("multiLine", True)
    .json(f"{RAW_BASE_PATH}/Condition/2026-08-14_2026-08-16/*.json")
)

print("Raw DataFrames loaded successfully")

In [0]:
from pyspark.sql.functions import col, explode, to_json


# ============================================================
# 1. Raw DataFrames for all FHIR resources
# ============================================================

raw_dataframes = {
    "Patient": patient_raw_df,
    "Encounter": encounter_raw_df,
    "Observation": observation_raw_df,
    "Condition": condition_raw_df
}


# ============================================================
# 2. Build Bronze test DataFrames
# ============================================================

bronze_test_dataframes = {}

for resource_type, raw_df in raw_dataframes.items():

    bronze_df = (
        raw_df
        .select(
            explode(col("entry")).alias("entry")
        )
        .select(
            col("entry.resource.resourceType").alias("resource_type"),
            col("entry.resource.id").alias("resource_id"),
            to_json(col("entry.resource")).alias("resource_json")
        )
    )

    bronze_test_dataframes[resource_type] = bronze_df


# ============================================================
# 3. Run all Bronze data-quality checks
# ============================================================

test_results = []

for resource_type, bronze_df in bronze_test_dataframes.items():

    # Raw record count
    raw_count = (
        raw_dataframes[resource_type]
        .select(explode(col("entry")).alias("entry"))
        .count()
    )

    # Bronze record count
    bronze_count = bronze_df.count()

    # Null IDs
    null_id_count = (
        bronze_df
        .filter(col("resource_id").isNull())
        .count()
    )

    # Duplicate IDs
    duplicate_id_count = (
        bronze_df
        .groupBy("resource_id")
        .count()
        .filter(col("count") > 1)
        .count()
    )

    # Invalid resource types
    invalid_resource_type_count = (
        bronze_df
        .filter(col("resource_type") != resource_type)
        .count()
    )

    # Null / empty JSON
    null_json_count = (
        bronze_df
        .filter(
            col("resource_json").isNull() |
            (col("resource_json") == "")
        )
        .count()
    )

    # Overall result
    all_tests_passed = (
        raw_count == bronze_count
        and null_id_count == 0
        and duplicate_id_count == 0
        and invalid_resource_type_count == 0
        and null_json_count == 0
    )

    test_results.append((
        resource_type,
        raw_count,
        bronze_count,
        null_id_count,
        duplicate_id_count,
        invalid_resource_type_count,
        null_json_count,
        "PASS" if all_tests_passed else "FAIL"
    ))


# ============================================================
# 4. Display consolidated test results
# ============================================================

test_results_df = spark.createDataFrame(
    test_results,
    [
        "resource_type",
        "raw_count",
        "bronze_count",
        "null_id_count",
        "duplicate_id_count",
        "invalid_resource_type_count",
        "null_json_count",
        "overall_result"
    ]
)

display(test_results_df)


# ============================================================
# 5. Fail the notebook if ANY test failed
# ============================================================

failed_tests = (
    test_results_df
    .filter(col("overall_result") == "FAIL")
    .count()
)

assert failed_tests == 0, (
    f"{failed_tests} resource(s) failed Bronze data-quality checks"
)

print("PASS: All Bronze data-quality checks passed for all resources")

In [0]:
%sql

SHOW TABLES IN fhir_assignment.bronze;

In [0]:
%sql

SELECT *
FROM fhir_assignment.bronze.patient
LIMIT 10;

In [0]:
%sql

SELECT 'Patient' AS resource, COUNT(*) AS records
FROM fhir_assignment.bronze.patient

UNION ALL

SELECT 'Encounter', COUNT(*)
FROM fhir_assignment.bronze.encounter

UNION ALL

SELECT 'Observation', COUNT(*)
FROM fhir_assignment.bronze.observation

UNION ALL

SELECT 'Condition', COUNT(*)
FROM fhir_assignment.bronze.condition;

In [0]:
%sql

DESCRIBE fhir_assignment.bronze.patient;



In [0]:
%sql
DESCRIBE fhir_assignment.bronze.encounter;


In [0]:
%sql
DESCRIBE fhir_assignment.bronze.observation;


In [0]:
%sql
DESCRIBE fhir_assignment.bronze.condition;

In [0]:
resources = [
    "patient",
    "encounter",
    "observation",
    "condition"
]

for resource in resources:
    print(f"\n{'=' * 80}")
    print(f"{resource.upper()}")
    print(f"{'=' * 80}")

    df = spark.table(f"fhir_assignment.bronze.{resource}")

    display(
        df.select(
            "resource_type",
            "resource_id",
            "resource_json"
        ).limit(1)
    )

In [0]:
from pyspark.sql.functions import col, get_json_object, count

observation_profile = (
    spark.table("fhir_assignment.bronze.observation")
    .select(
        get_json_object(col("resource_json"), "$.valueQuantity")
            .alias("value_quantity"),

        get_json_object(col("resource_json"), "$.valueString")
            .alias("value_string"),

        get_json_object(col("resource_json"), "$.valueCodeableConcept")
            .alias("value_codeable_concept"),

        get_json_object(col("resource_json"), "$.valueBoolean")
            .alias("value_boolean")
    )
)

display(
    observation_profile.select(
        count("*").alias("total_records"),
        count("value_quantity").alias("value_quantity_records"),
        count("value_string").alias("value_string_records"),
        count("value_codeable_concept").alias("value_codeable_concept_records"),
        count("value_boolean").alias("value_boolean_records")
    )
)

In [0]:
from pyspark.sql.functions import col, get_json_object, count

# ============================================================
# Profile important FHIR fields across all resources
# ============================================================

profiles = {}

# -------------------------
# Patient
# -------------------------
patient_df = spark.table("fhir_assignment.bronze.patient")

profiles["Patient"] = patient_df.select(
    count("*").alias("total_records"),
    count(get_json_object(col("resource_json"), "$.gender")).alias("gender"),
    count(get_json_object(col("resource_json"), "$.birthDate")).alias("birth_date"),
    count(get_json_object(col("resource_json"), "$.name")).alias("name"),
    count(get_json_object(col("resource_json"), "$.identifier")).alias("identifier"),
    count(get_json_object(col("resource_json"), "$.address")).alias("address")
)


# -------------------------
# Encounter
# -------------------------
encounter_df = spark.table("fhir_assignment.bronze.encounter")

profiles["Encounter"] = encounter_df.select(
    count("*").alias("total_records"),
    count(get_json_object(col("resource_json"), "$.status")).alias("status"),
    count(get_json_object(col("resource_json"), "$.class")).alias("class"),
    count(get_json_object(col("resource_json"), "$.subject")).alias("subject"),
    count(get_json_object(col("resource_json"), "$.period")).alias("period"),
    count(get_json_object(col("resource_json"), "$.participant")).alias("participant"),
    count(get_json_object(col("resource_json"), "$.serviceProvider")).alias("service_provider")
)


# -------------------------
# Observation
# -------------------------
observation_df = spark.table("fhir_assignment.bronze.observation")

profiles["Observation"] = observation_df.select(
    count("*").alias("total_records"),
    count(get_json_object(col("resource_json"), "$.status")).alias("status"),
    count(get_json_object(col("resource_json"), "$.category")).alias("category"),
    count(get_json_object(col("resource_json"), "$.code")).alias("code"),
    count(get_json_object(col("resource_json"), "$.subject")).alias("subject"),
    count(get_json_object(col("resource_json"), "$.effectiveDateTime")).alias("effective_datetime"),
    count(get_json_object(col("resource_json"), "$.valueQuantity")).alias("value_quantity"),
    count(get_json_object(col("resource_json"), "$.valueString")).alias("value_string"),
    count(get_json_object(col("resource_json"), "$.valueCodeableConcept")).alias("value_codeable_concept")
)


# -------------------------
# Condition
# -------------------------
condition_df = spark.table("fhir_assignment.bronze.condition")

profiles["Condition"] = condition_df.select(
    count("*").alias("total_records"),
    count(get_json_object(col("resource_json"), "$.clinicalStatus")).alias("clinical_status"),
    count(get_json_object(col("resource_json"), "$.code")).alias("code"),
    count(get_json_object(col("resource_json"), "$.subject")).alias("subject"),
    count(get_json_object(col("resource_json"), "$.encounter")).alias("encounter"),
    count(get_json_object(col("resource_json"), "$.onsetDateTime")).alias("onset_date")
)


# Display all profiles
for resource, profile in profiles.items():
    print(f"\n{'=' * 70}")
    print(resource.upper())
    print(f"{'=' * 70}")
    display(profile)

In [0]:
from pyspark.sql.functions import col, get_json_object

# ============================================================
# Profile important nested FHIR fields across all Bronze tables
# ============================================================

bronze_tables = {
    "Patient": "fhir_assignment.bronze.patient",
    "Encounter": "fhir_assignment.bronze.encounter",
    "Observation": "fhir_assignment.bronze.observation",
    "Condition": "fhir_assignment.bronze.condition"
}

# JSON fields we want to inspect
field_profiles = {
    "Patient": [
        ("gender", "$.gender"),
        ("birth_date", "$.birthDate"),
        ("name", "$.name"),
        ("identifier", "$.identifier"),
        ("address", "$.address")
    ],

    "Encounter": [
        ("status", "$.status"),
        ("class", "$.class"),
        ("subject", "$.subject"),
        ("period", "$.period"),
        ("participant", "$.participant"),
        ("service_provider", "$.serviceProvider")
    ],

    "Observation": [
        ("status", "$.status"),
        ("category", "$.category"),
        ("code", "$.code"),
        ("subject", "$.subject"),
        ("effective_datetime", "$.effectiveDateTime"),
        ("value_quantity", "$.valueQuantity"),
        ("value_string", "$.valueString")
    ],

    "Condition": [
        ("clinical_status", "$.clinicalStatus"),
        ("code", "$.code"),
        ("subject", "$.subject"),
        ("encounter", "$.encounter"),
        ("onset_date", "$.onsetDateTime")
    ]
}


for resource, table_name in bronze_tables.items():

    print("\n" + "=" * 70)
    print(resource.upper())
    print("=" * 70)

    df = spark.table(table_name)

    # Build profile columns
    profile_columns = [
        col("resource_id")
    ]

    for field_name, json_path in field_profiles[resource]:
        profile_columns.append(
            get_json_object(
                col("resource_json"),
                json_path
            ).alias(field_name)
        )

    profile_df = df.select(*profile_columns)

    # Show a few actual records
    display(profile_df.limit(5))

# Silver Testing and Data Quality Check

In [0]:
from pyspark.sql.functions import (
    col,
    get_json_object,
    to_date,
    to_timestamp
)

# ============================================================
# SILVER TRANSFORMATION
# Bronze -> Silver
# ============================================================


# ============================================================
# 1. PATIENT
# ============================================================

patient_silver = (
    spark.table("fhir_assignment.bronze.patient")
    .select(
        col("resource_id").alias("patient_id"),

        get_json_object(
            col("resource_json"), "$.gender"
        ).alias("gender"),

        to_date(
            get_json_object(
                col("resource_json"), "$.birthDate"
            )
        ).alias("birth_date"),

        get_json_object(
            col("resource_json"), "$.name[0].family"
        ).alias("family_name"),

        get_json_object(
            col("resource_json"), "$.name[0].given[0]"
        ).alias("given_name"),

        get_json_object(
            col("resource_json"), "$.name[0].text"
        ).alias("name_text"),

        get_json_object(
            col("resource_json"), "$.identifier[0].value"
        ).alias("identifier_value"),

        get_json_object(
            col("resource_json"), "$.address[0].city"
        ).alias("city"),

        get_json_object(
            col("resource_json"), "$.address[0].state"
        ).alias("state"),

        get_json_object(
            col("resource_json"), "$.address[0].postalCode"
        ).alias("postal_code"),

        col("source_file"),
        col("ingestion_timestamp")
    )
)


# ============================================================
# 2. ENCOUNTER
# ============================================================

encounter_silver = (
    spark.table("fhir_assignment.bronze.encounter")
    .select(
        col("resource_id").alias("encounter_id"),

        get_json_object(
            col("resource_json"), "$.status"
        ).alias("status"),

        get_json_object(
            col("resource_json"), "$.class.code"
        ).alias("class_code"),

        get_json_object(
            col("resource_json"), "$.class.display"
        ).alias("class_display"),

        get_json_object(
            col("resource_json"), "$.subject.reference"
        ).alias("patient_reference"),

        to_timestamp(
            get_json_object(
                col("resource_json"), "$.period.start"
            )
        ).alias("period_start"),

        to_timestamp(
            get_json_object(
                col("resource_json"), "$.period.end"
            )
        ).alias("period_end"),

        get_json_object(
            col("resource_json"),
            "$.participant[0].individual.reference"
        ).alias("practitioner_reference"),

        get_json_object(
            col("resource_json"),
            "$.participant[0].individual.display"
        ).alias("practitioner_name"),

        get_json_object(
            col("resource_json"),
            "$.serviceProvider.display"
        ).alias("service_provider"),

        get_json_object(
            col("resource_json"),
            "$.serviceProvider.identifier.value"
        ).alias("facility_id"),

        col("source_file"),
        col("ingestion_timestamp")
    )
)


# ============================================================
# 3. OBSERVATION
# ============================================================

observation_silver = (
    spark.table("fhir_assignment.bronze.observation")
    .select(
        col("resource_id").alias("observation_id"),

        get_json_object(
            col("resource_json"), "$.status"
        ).alias("status"),

        get_json_object(
            col("resource_json"),
            "$.category[0].coding[0].code"
        ).alias("category_code"),

        get_json_object(
            col("resource_json"),
            "$.code.coding[0].code"
        ).alias("observation_code"),

        get_json_object(
            col("resource_json"),
            "$.code.coding[0].display"
        ).alias("observation_display"),

        get_json_object(
            col("resource_json"),
            "$.subject.reference"
        ).alias("patient_reference"),

        to_timestamp(
            get_json_object(
                col("resource_json"),
                "$.effectiveDateTime"
            )
        ).alias("effective_datetime"),

        get_json_object(
            col("resource_json"),
            "$.valueQuantity.value"
        ).cast("double").alias("value_quantity"),

        get_json_object(
            col("resource_json"),
            "$.valueQuantity.unit"
        ).alias("value_unit"),

        get_json_object(
            col("resource_json"),
            "$.valueString"
        ).alias("value_string"),

        # Keep other FHIR value types so we don't lose information
        get_json_object(
            col("resource_json"),
            "$.valueCodeableConcept.text"
        ).alias("value_codeable_text"),

        col("source_file"),
        col("ingestion_timestamp")
    )
)


# ============================================================
# 4. CONDITION
# ============================================================

condition_silver = (
    spark.table("fhir_assignment.bronze.condition")
    .select(
        col("resource_id").alias("condition_id"),

        get_json_object(
            col("resource_json"),
            "$.clinicalStatus.coding[0].code"
        ).alias("clinical_status"),

        get_json_object(
            col("resource_json"),
            "$.code.coding[0].code"
        ).alias("condition_code"),

        get_json_object(
            col("resource_json"),
            "$.code.coding[0].display"
        ).alias("condition_display"),

        get_json_object(
            col("resource_json"),
            "$.code.text"
        ).alias("condition_text"),

        get_json_object(
            col("resource_json"),
            "$.subject.reference"
        ).alias("patient_reference"),

        get_json_object(
            col("resource_json"),
            "$.encounter.reference"
        ).alias("encounter_reference"),

        to_date(
            get_json_object(
                col("resource_json"),
                "$.onsetDateTime"
            )
        ).alias("onset_date"),

        col("source_file"),
        col("ingestion_timestamp")
    )
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("PATIENT SILVER")
display(patient_silver.limit(10))

print("ENCOUNTER SILVER")
display(encounter_silver.limit(10))

print("OBSERVATION SILVER")
display(observation_silver.limit(10))

print("CONDITION SILVER")
display(condition_silver.limit(10))

In [0]:
from pyspark.sql.functions import col, count, when


# ============================================================
# SILVER DATA QUALITY TESTS
# ============================================================

silver_dataframes = {
    "Patient": (
        patient_silver,
        "patient_id"
    ),
    "Encounter": (
        encounter_silver,
        "encounter_id"
    ),
    "Observation": (
        observation_silver,
        "observation_id"
    ),
    "Condition": (
        condition_silver,
        "condition_id"
    )
}


results = []


for resource, (silver_df, primary_key) in silver_dataframes.items():

    bronze_table = f"fhir_assignment.bronze.{resource.lower()}"

    bronze_count = spark.table(bronze_table).count()
    silver_count = silver_df.count()

    null_id_count = (
        silver_df
        .filter(col(primary_key).isNull())
        .count()
    )

    duplicate_id_count = (
        silver_df
        .groupBy(primary_key)
        .count()
        .filter(col("count") > 1)
        .count()
    )

    results.append({
        "resource": resource,
        "bronze_count": bronze_count,
        "silver_count": silver_count,
        "null_id_count": null_id_count,
        "duplicate_id_count": duplicate_id_count
    })


# Convert results to DataFrame
results_df = spark.createDataFrame(results)

display(results_df)

In [0]:
# ============================================================
# SILVER REFERENTIAL INTEGRITY TESTS
# ============================================================

# Remove "Patient/" and "Encounter/" prefixes from references
# so they can be compared with the Silver primary keys.

encounter_check = (
    encounter_silver
    .withColumn(
        "patient_id",
        col("patient_reference").substr(
            9, 1000
        )
    )
)

observation_check = (
    observation_silver
    .withColumn(
        "patient_id",
        col("patient_reference").substr(
            9, 1000
        )
    )
)

condition_check = (
    condition_silver
    .withColumn(
        "patient_id",
        col("patient_reference").substr(
            9, 1000
        )
    )
    .withColumn(
        "encounter_id",
        col("encounter_reference").substr(
            11, 1000
        )
    )
)


# ============================================================
# Create lookup sets
# ============================================================

patient_ids = (
    patient_silver
    .select("patient_id")
    .distinct()
)

encounter_ids = (
    encounter_silver
    .select("encounter_id")
    .distinct()
)


# ============================================================
# 1. Encounter → Patient
# ============================================================

encounter_orphans = (
    encounter_check
    .filter(col("patient_id").isNotNull())
    .join(
        patient_ids,
        on="patient_id",
        how="left_anti"
    )
)

encounter_orphan_count = encounter_orphans.count()


# ============================================================
# 2. Observation → Patient
# ============================================================

observation_orphans = (
    observation_check
    .filter(col("patient_id").isNotNull())
    .join(
        patient_ids,
        on="patient_id",
        how="left_anti"
    )
)

observation_orphan_count = observation_orphans.count()


# ============================================================
# 3. Condition → Patient
# ============================================================

condition_patient_orphans = (
    condition_check
    .filter(col("patient_id").isNotNull())
    .join(
        patient_ids,
        on="patient_id",
        how="left_anti"
    )
)

condition_patient_orphan_count = condition_patient_orphans.count()


# ============================================================
# 4. Condition → Encounter
# ============================================================

condition_encounter_orphans = (
    condition_check
    .filter(col("encounter_id").isNotNull())
    .join(
        encounter_ids,
        on="encounter_id",
        how="left_anti"
    )
)

condition_encounter_orphan_count = (
    condition_encounter_orphans.count()
)


# ============================================================
# Display results
# ============================================================

relationship_results = [
    (
        "Encounter → Patient",
        encounter_orphan_count
    ),
    (
        "Observation → Patient",
        observation_orphan_count
    ),
    (
        "Condition → Patient",
        condition_patient_orphan_count
    ),
    (
        "Condition → Encounter",
        condition_encounter_orphan_count
    )
]

relationship_results_df = spark.createDataFrame(
    relationship_results,
    ["relationship", "orphan_count"]
)

display(relationship_results_df)


# ============================================================
# Final assertion
# ============================================================

total_orphans = (
    encounter_orphan_count
    + observation_orphan_count
    + condition_patient_orphan_count
    + condition_encounter_orphan_count
)

assert total_orphans == 0, (
    f"Found {total_orphans} orphan references"
)

print(
    "PASS: All Silver referential-integrity checks passed"
)

In [0]:
# ============================================================
# INSPECT ORPHAN REFERENCES
# ============================================================

print("ENCOUNTER → PATIENT ORPHANS")
display(
    encounter_orphans
    .select(
        "encounter_id",
        "patient_reference",
        "patient_id"
    )
    .limit(20)
)


print("OBSERVATION → PATIENT ORPHANS")
display(
    observation_orphans
    .select(
        "observation_id",
        "patient_reference",
        "patient_id"
    )
    .limit(20)
)


print("CONDITION → PATIENT ORPHANS")
display(
    condition_patient_orphans
    .select(
        "condition_id",
        "patient_reference",
        "patient_id"
    )
    .limit(20)
)


print("CONDITION → ENCOUNTER ORPHANS")
display(
    condition_encounter_orphans
    .select(
        "condition_id",
        "encounter_reference",
        "encounter_id"
    )
    .limit(20)
)

In [0]:
# ============================================================
# SILVER REFERENTIAL INTEGRITY REPORT
# ============================================================

relationship_results = [
    (
        "Encounter → Patient",
        encounter_orphan_count
    ),
    (
        "Observation → Patient",
        observation_orphan_count
    ),
    (
        "Condition → Patient",
        condition_patient_orphan_count
    ),
    (
        "Condition → Encounter",
        condition_encounter_orphan_count
    )
]

relationship_results_df = spark.createDataFrame(
    relationship_results,
    ["relationship", "unresolved_reference_count"]
)

display(relationship_results_df)

print(
    "PASS: Referential-integrity check completed. "
    "Unresolved references are retained and reported."
)

In [0]:
# ============================================================
# SILVER TYPE & DATE VALIDATION
# ============================================================

silver_dataframes = {
    "Patient": patient_silver,
    "Encounter": encounter_silver,
    "Observation": observation_silver,
    "Condition": condition_silver
}

for resource, df in silver_dataframes.items():

    print("\n" + "=" * 70)
    print(resource.upper())
    print("=" * 70)

    df.printSchema()

In [0]:
# ============================================================
# DATE / TIMESTAMP VALIDATION
# ============================================================

date_tests = {
    "Patient.birth_date": (
        patient_silver,
        "birth_date"
    ),

    "Encounter.period_start": (
        encounter_silver,
        "period_start"
    ),

    "Encounter.period_end": (
        encounter_silver,
        "period_end"
    ),

    "Observation.effective_datetime": (
        observation_silver,
        "effective_datetime"
    ),

    "Condition.onset_date": (
        condition_silver,
        "onset_date"
    )
}

for field_name, (df, column_name) in date_tests.items():

    print(
        field_name,
        "->",
        df.schema[column_name].dataType
    )

In [0]:
%sql

SHOW TABLES IN fhir_assignment.silver;

In [0]:
%sql

SELECT 'Patient' AS resource, COUNT(*) AS silver_count
FROM fhir_assignment.silver.patient

UNION ALL

SELECT 'Encounter', COUNT(*)
FROM fhir_assignment.silver.encounter

UNION ALL

SELECT 'Observation', COUNT(*)
FROM fhir_assignment.silver.observation

UNION ALL

SELECT 'Condition', COUNT(*)
FROM fhir_assignment.silver.condition;

# Gold Testing and Data Quality Check

In [0]:
# ============================================================
# GOLD DATA PROFILING
# ============================================================

from pyspark.sql.functions import (
    col,
    count,
    min,
    max,
    avg
)

gold_sources = {
    "Patient": patient_silver,
    "Encounter": encounter_silver,
    "Observation": observation_silver,
    "Condition": condition_silver
}

for resource, df in gold_sources.items():

    print("\n" + "=" * 70)
    print(resource.upper())
    print("=" * 70)

    print("Record count:", df.count())

    display(df.limit(10))

In [0]:
from pyspark.sql.functions import (
    col,
    concat_ws,
    coalesce,
    lit,
    to_date,
    round,
    when
)

# ============================================================
# READ CURRENT SILVER TABLES
# ============================================================

patient_silver = spark.table("fhir_assignment.silver.patient")
encounter_silver = spark.table("fhir_assignment.silver.encounter")
observation_silver = spark.table("fhir_assignment.silver.observation")
condition_silver = spark.table("fhir_assignment.silver.condition")


# ============================================================
# GOLD 1: DIM PATIENT
# ============================================================

dim_patient = (
    patient_silver
    .select(
        "patient_id",
        "gender",
        "birth_date",
        "family_name",
        "given_name",
        "name_text",
        "identifier_value",
        "city",
        "state",
        "postal_code",
        "source_file",
        "ingestion_timestamp"
    )
    .withColumn(
        "patient_name",
        coalesce(
            concat_ws(" ", col("given_name"), col("family_name")),
            col("name_text")
        )
    )
    .select(
        "patient_id",
        "patient_name",
        "gender",
        "birth_date",
        "identifier_value",
        "city",
        "state",
        "postal_code",
        "source_file",
        "ingestion_timestamp"
    )
)


# ============================================================
# GOLD 2: FACT ENCOUNTER
# ============================================================

fact_encounter = (
    encounter_silver
    .select(
        "encounter_id",
        "patient_id",
        "status",
        "class_code",
        "class_display",
        "period_start",
        "period_end",
        "practitioner_id",
        "practitioner_name",
        "service_provider",
        "facility_id",
        "source_file",
        "ingestion_timestamp"
    )
    .withColumn(
        "encounter_date",
        to_date(col("period_start"))
    )
    .withColumn(
        "duration_hours",
        when(
            col("period_start").isNotNull()
            & col("period_end").isNotNull(),
            round(
                (
                    col("period_end").cast("long")
                    - col("period_start").cast("long")
                ) / 3600.0,
                2
            )
        )
    )
    .select(
        "encounter_id",
        "patient_id",
        "encounter_date",
        "status",
        "class_code",
        "class_display",
        "period_start",
        "period_end",
        "duration_hours",
        "practitioner_id",
        "practitioner_name",
        "service_provider",
        "facility_id",
        "source_file",
        "ingestion_timestamp"
    )
)


# ============================================================
# GOLD 3: FACT OBSERVATION
# ============================================================

fact_observation = (
    observation_silver
    .select(
        "observation_id",
        "patient_id",
        "status",
        "category_code",
        "observation_code",
        "observation_display",
        "effective_datetime",
        "value_quantity",
        "value_unit",
        "value_string",
        "value_codeable_text",
        "source_file",
        "ingestion_timestamp"
    )
    .withColumn(
        "observation_date",
        to_date(col("effective_datetime"))
    )
    .withColumn(
        "value_type",
        when(
            col("value_quantity").isNotNull(),
            lit("Quantity")
        )
        .when(
            col("value_string").isNotNull(),
            lit("String")
        )
        .when(
            col("value_codeable_text").isNotNull(),
            lit("CodeableConcept")
        )
        .otherwise(
            lit("Unknown")
        )
    )
    .select(
        "observation_id",
        "patient_id",
        "observation_date",
        "effective_datetime",
        "status",
        "category_code",
        "observation_code",
        "observation_display",
        "value_quantity",
        "value_unit",
        "value_string",
        "value_codeable_text",
        "value_type",
        "source_file",
        "ingestion_timestamp"
    )
)


# ============================================================
# GOLD 4: FACT CONDITION
# ============================================================

fact_condition = (
    condition_silver
    .select(
        "condition_id",
        "patient_id",
        "encounter_id",
        "clinical_status",
        "condition_code",
        "condition_display",
        "condition_text",
        "onset_date",
        "source_file",
        "ingestion_timestamp"
    )
)


# ============================================================
# COUNT CHECK
# ============================================================

print("=" * 70)
print("GOLD DATAFRAMES CREATED")
print("=" * 70)

print(f"dim_patient      : {dim_patient.count()}")
print(f"fact_encounter   : {fact_encounter.count()}")
print(f"fact_observation : {fact_observation.count()}")
print(f"fact_condition   : {fact_condition.count()}")


# ============================================================
# SAMPLE DATA
# ============================================================

print("\nDIM PATIENT")
display(dim_patient.limit(10))

print("\nFACT ENCOUNTER")
display(fact_encounter.limit(10))

print("\nFACT OBSERVATION")
display(fact_observation.limit(10))

print("\nFACT CONDITION")
display(fact_condition.limit(10))

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum as spark_sum,
    when,
    lit
)

# ============================================================
# GOLD DATA QUALITY VALIDATION
# ============================================================

print("=" * 80)
print("GOLD DATA QUALITY VALIDATION")
print("=" * 80)


# ============================================================
# 1. RECORD COUNT VALIDATION
# ============================================================

gold_counts = [
    ("Patient", dim_patient.count(), 261),
    ("Encounter", fact_encounter.count(), 92),
    ("Observation", fact_observation.count(), 4578),
    ("Condition", fact_condition.count(), 48)
]

print("\nRECORD COUNTS")
print("-" * 80)

for resource, actual, expected in gold_counts:
    result = "PASS" if actual == expected else "FAIL"

    print(
        f"{resource:<15} "
        f"Actual={actual:<6} "
        f"Expected={expected:<6} "
        f"{result}"
    )

    assert actual == expected, (
        f"{resource} count mismatch: "
        f"actual={actual}, expected={expected}"
    )


# ============================================================
# 2. PRIMARY KEY NULL CHECKS
# ============================================================

print("\nPRIMARY KEY NULL CHECKS")
print("-" * 80)

pk_checks = {
    "Patient": (dim_patient, "patient_id"),
    "Encounter": (fact_encounter, "encounter_id"),
    "Observation": (fact_observation, "observation_id"),
    "Condition": (fact_condition, "condition_id")
}

for resource, (df, pk) in pk_checks.items():

    null_count = (
        df.filter(
            col(pk).isNull() | (col(pk) == "")
        )
        .count()
    )

    result = "PASS" if null_count == 0 else "FAIL"

    print(
        f"{resource:<15} "
        f"{pk:<20} "
        f"Null/empty={null_count:<5} "
        f"{result}"
    )

    assert null_count == 0, (
        f"{resource} contains {null_count} "
        f"null/empty {pk} values"
    )


# ============================================================
# 3. DUPLICATE PRIMARY KEY CHECK
# ============================================================

print("\nDUPLICATE PRIMARY KEY CHECKS")
print("-" * 80)

for resource, (df, pk) in pk_checks.items():

    duplicate_count = (
        df.groupBy(pk)
        .count()
        .filter(col("count") > 1)
        .count()
    )

    result = "PASS" if duplicate_count == 0 else "FAIL"

    print(
        f"{resource:<15} "
        f"Duplicate {pk}={duplicate_count:<5} "
        f"{result}"
    )

    assert duplicate_count == 0, (
        f"{resource} contains duplicate {pk} values"
    )


# ============================================================
# 4. DATE / TIMESTAMP NULL PROFILE
# ============================================================

print("\nDATE / TIMESTAMP PROFILE")
print("-" * 80)

date_profiles = {
    "Patient.birth_date": (dim_patient, "birth_date"),
    "Encounter.period_start": (fact_encounter, "period_start"),
    "Encounter.period_end": (fact_encounter, "period_end"),
    "Observation.effective_datetime": (
        fact_observation,
        "effective_datetime"
    ),
    "Condition.onset_date": (fact_condition, "onset_date")
}

for field, (df, column_name) in date_profiles.items():

    total = df.count()

    null_count = (
        df.filter(col(column_name).isNull())
        .count()
    )

    populated_count = total - null_count

    print(
        f"{field:<40} "
        f"Populated={populated_count:<6} "
        f"Null={null_count:<6}"
    )


# ============================================================
# 5. ENCOUNTER DURATION VALIDATION
# ============================================================

print("\nENCOUNTER DURATION VALIDATION")
print("-" * 80)

negative_duration_count = (
    fact_encounter
    .filter(
        col("duration_hours").isNotNull()
        & (col("duration_hours") < 0)
    )
    .count()
)

print(
    f"Negative duration records: "
    f"{negative_duration_count}"
)

assert negative_duration_count == 0, (
    f"Found {negative_duration_count} "
    f"negative encounter durations"
)

print("PASS: No negative encounter durations")


# ============================================================
# 6. OBSERVATION VALUE TYPE VALIDATION
# ============================================================

print("\nOBSERVATION VALUE TYPE VALIDATION")
print("-" * 80)

display(
    fact_observation
    .groupBy("value_type")
    .count()
    .orderBy("value_type")
)


# ============================================================
# 7. GOLD RELATIONSHIP PROFILE
# ============================================================

print("\nGOLD RELATIONSHIP PROFILE")
print("-" * 80)

patient_ids = (
    dim_patient
    .select("patient_id")
    .distinct()
)

encounter_ids = (
    fact_encounter
    .select("encounter_id")
    .distinct()
)


# Encounter -> Patient
encounter_orphans = (
    fact_encounter.alias("e")
    .join(
        patient_ids.alias("p"),
        col("e.patient_id") == col("p.patient_id"),
        "left"
    )
    .filter(
        col("e.patient_id").isNotNull()
        & col("p.patient_id").isNull()
    )
    .count()
)


# Observation -> Patient
observation_orphans = (
    fact_observation.alias("o")
    .join(
        patient_ids.alias("p"),
        col("o.patient_id") == col("p.patient_id"),
        "left"
    )
    .filter(
        col("o.patient_id").isNotNull()
        & col("p.patient_id").isNull()
    )
    .count()
)


# Condition -> Patient
condition_patient_orphans = (
    fact_condition.alias("c")
    .join(
        patient_ids.alias("p"),
        col("c.patient_id") == col("p.patient_id"),
        "left"
    )
    .filter(
        col("c.patient_id").isNotNull()
        & col("p.patient_id").isNull()
    )
    .count()
)


# Condition -> Encounter
condition_encounter_orphans = (
    fact_condition.alias("c")
    .join(
        encounter_ids.alias("e"),
        col("c.encounter_id") == col("e.encounter_id"),
        "left"
    )
    .filter(
        col("c.encounter_id").isNotNull()
        & col("e.encounter_id").isNull()
    )
    .count()
)


print(
    f"Encounter -> Patient : {encounter_orphans}"
)

print(
    f"Observation -> Patient : {observation_orphans}"
)

print(
    f"Condition -> Patient : {condition_patient_orphans}"
)

print(
    f"Condition -> Encounter : {condition_encounter_orphans}"
)


# ============================================================
# 8. DATA QUALITY SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("GOLD DATA QUALITY SUMMARY")
print("=" * 80)

print("PASS: Record counts match expected Silver counts")
print("PASS: Primary keys contain no null/empty values")
print("PASS: Primary keys contain no duplicates")
print("PASS: No negative encounter durations")
print("PASS: Gold transformations completed successfully")

print("\nReferential integrity is reported separately because")
print("the source FHIR dataset contains unresolved references.")

print("=" * 80)

In [0]:
# ============================================================
# GOLD TABLE POST-WRITE VERIFICATION
# ============================================================

gold_tables = {
    "Patient": "fhir_assignment.gold.dim_patient",
    "Encounter": "fhir_assignment.gold.fact_encounter",
    "Observation": "fhir_assignment.gold.fact_observation",
    "Condition": "fhir_assignment.gold.fact_condition"
}

for resource, table_name in gold_tables.items():

    print("=" * 80)
    print(resource.upper())
    print("=" * 80)

    df = spark.table(table_name)

    print(f"Record count: {df.count()}")

    display(df.limit(10))

In [0]:
# ============================================================
# TESTING CONFIGURATION
# ============================================================

CATALOG = "fhir_assignment"
GOLD_SCHEMA = "gold"

print(f"Catalog    : {CATALOG}")
print(f"Gold schema: {GOLD_SCHEMA}")



# ============================================================
# FINAL GOLD DATA QUALITY VALIDATION
# ============================================================

from pyspark.sql import functions as F

print("=" * 80)
print("GOLD DATA QUALITY VALIDATION")
print("=" * 80)


# ============================================================
# 1. LOAD GOLD TABLES
# ============================================================

dim_patient = spark.table(
    f"{CATALOG}.{GOLD_SCHEMA}.dim_patient"
)

fact_encounter = spark.table(
    f"{CATALOG}.{GOLD_SCHEMA}.fact_encounter"
)

fact_observation = spark.table(
    f"{CATALOG}.{GOLD_SCHEMA}.fact_observation"
)

fact_condition = spark.table(
    f"{CATALOG}.{GOLD_SCHEMA}.fact_condition"
)


# ============================================================
# 2. EXPECTED RECORD COUNTS
# ============================================================

expected_counts = {
    "Patient": 261,
    "Encounter": 92,
    "Observation": 4578,
    "Condition": 48
}

actual_counts = {
    "Patient": dim_patient.count(),
    "Encounter": fact_encounter.count(),
    "Observation": fact_observation.count(),
    "Condition": fact_condition.count()
}

print("\nRECORD COUNTS")
print("-" * 80)

for resource in expected_counts:

    actual = actual_counts[resource]
    expected = expected_counts[resource]

    status = "PASS" if actual == expected else "FAIL"

    print(
        f"{resource:<15} "
        f"Actual={actual:<6} "
        f"Expected={expected:<6} "
        f"{status}"
    )


# ============================================================
# 3. PRIMARY KEY NULL / EMPTY CHECK
# ============================================================

print("\nPRIMARY KEY NULL CHECKS")
print("-" * 80)


pk_columns = {
    "Patient": (dim_patient, "patient_id"),
    "Encounter": (fact_encounter, "encounter_id"),
    "Observation": (fact_observation, "observation_id"),
    "Condition": (fact_condition, "condition_id")
}

pk_null_failures = 0

for resource, (df, pk) in pk_columns.items():

    null_count = (
        df
        .filter(
            F.col(pk).isNull() |
            (F.trim(F.col(pk)) == "")
        )
        .count()
    )

    status = "PASS" if null_count == 0 else "FAIL"

    print(
        f"{resource:<15} "
        f"{pk:<20} "
        f"Null/empty={null_count:<6} "
        f"{status}"
    )

    pk_null_failures += null_count


# ============================================================
# 4. DUPLICATE PRIMARY KEY CHECK
# ============================================================

print("\nDUPLICATE PRIMARY KEY CHECKS")
print("-" * 80)

pk_duplicate_failures = 0

for resource, (df, pk) in pk_columns.items():

    duplicate_count = (
        df
        .groupBy(pk)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    status = "PASS" if duplicate_count == 0 else "FAIL"

    print(
        f"{resource:<15} "
        f"Duplicate {pk}={duplicate_count:<6} "
        f"{status}"
    )

    pk_duplicate_failures += duplicate_count


# ============================================================
# 5. DATE / TIMESTAMP PROFILE
# ============================================================

print("\nDATE / TIMESTAMP PROFILE")
print("-" * 80)


date_profiles = {
    "Patient.birth_date": dim_patient,
    "Encounter.period_start": fact_encounter,
    "Encounter.period_end": fact_encounter,
    "Observation.effective_datetime": fact_observation,
    "Condition.onset_date": fact_condition
}

date_columns = {
    "Patient.birth_date": "birth_date",
    "Encounter.period_start": "period_start",
    "Encounter.period_end": "period_end",
    "Observation.effective_datetime": "effective_datetime",
    "Condition.onset_date": "onset_date"
}

for name, df in date_profiles.items():

    column = date_columns[name]

    populated = (
        df
        .filter(F.col(column).isNotNull())
        .count()
    )

    total = df.count()
    null_count = total - populated

    print(
        f"{name:<40} "
        f"Populated={populated:<6} "
        f"Null={null_count:<6}"
    )


# ============================================================
# 6. ENCOUNTER DURATION VALIDATION
# ============================================================

print("\nENCOUNTER DURATION VALIDATION")
print("-" * 80)

negative_duration_count = (
    fact_encounter
    .filter(
        F.col("duration_hours").isNotNull() &
        (F.col("duration_hours") < 0)
    )
    .count()
)

print(
    f"Negative duration records: "
    f"{negative_duration_count}"
)

if negative_duration_count == 0:
    print("PASS: No negative encounter durations")
else:
    print("FAIL: Negative encounter durations found")


# ============================================================
# 7. OBSERVATION VALUE TYPE PROFILE
# ============================================================

print("\nOBSERVATION VALUE TYPE VALIDATION")
print("-" * 80)

value_type_profile = (
    fact_observation
    .groupBy("value_type")
    .count()
    .orderBy("value_type")
)

display(value_type_profile)


# ============================================================
# 8. REFERENTIAL INTEGRITY PROFILE
# ============================================================
#
# IMPORTANT:
# We intentionally DO NOT fail the pipeline on these.
#
# The source FHIR dataset contains references to Patient /
# Encounter resources that are not present in this dataset.
#
# We report them separately instead.
# ============================================================

print("\nGOLD RELATIONSHIP PROFILE")
print("-" * 80)


# ----------------------------
# Encounter -> Patient
# ----------------------------

encounter_orphans = (
    fact_encounter.alias("e")
    .join(
        dim_patient.alias("p"),
        F.col("e.patient_id") == F.col("p.patient_id"),
        "left"
    )
    .filter(
        F.col("e.patient_id").isNotNull() &
        F.col("p.patient_id").isNull()
    )
    .count()
)

print(
    f"Encounter -> Patient : "
    f"{encounter_orphans}"
)


# ----------------------------
# Observation -> Patient
# ----------------------------

observation_orphans = (
    fact_observation.alias("o")
    .join(
        dim_patient.alias("p"),
        F.col("o.patient_id") == F.col("p.patient_id"),
        "left"
    )
    .filter(
        F.col("o.patient_id").isNotNull() &
        F.col("p.patient_id").isNull()
    )
    .count()
)

print(
    f"Observation -> Patient : "
    f"{observation_orphans}"
)


# ----------------------------
# Condition -> Patient
# ----------------------------

condition_patient_orphans = (
    fact_condition.alias("c")
    .join(
        dim_patient.alias("p"),
        F.col("c.patient_id") == F.col("p.patient_id"),
        "left"
    )
    .filter(
        F.col("c.patient_id").isNotNull() &
        F.col("p.patient_id").isNull()
    )
    .count()
)

print(
    f"Condition -> Patient : "
    f"{condition_patient_orphans}"
)


# ----------------------------
# Condition -> Encounter
# ----------------------------

condition_encounter_orphans = (
    fact_condition.alias("c")
    .join(
        fact_encounter.alias("e"),
        F.col("c.encounter_id") == F.col("e.encounter_id"),
        "left"
    )
    .filter(
        F.col("c.encounter_id").isNotNull() &
        F.col("e.encounter_id").isNull()
    )
    .count()
)

print(
    f"Condition -> Encounter : "
    f"{condition_encounter_orphans}"
)


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("GOLD DATA QUALITY SUMMARY")
print("=" * 80)


count_failures = sum(
    actual_counts[r] != expected_counts[r]
    for r in expected_counts
)

duration_failure = negative_duration_count > 0

if (
    count_failures == 0
    and pk_null_failures == 0
    and pk_duplicate_failures == 0
    and not duration_failure
):

    print("PASS: Record counts match expected Silver counts")
    print("PASS: Primary keys contain no null/empty values")
    print("PASS: Primary keys contain no duplicates")
    print("PASS: No negative encounter durations")
    print("PASS: Gold transformations completed successfully")

else:

    print("FAIL: One or more Gold data quality checks failed")


print()
print(
    "Referential integrity is reported separately because"
)
print(
    "the source FHIR dataset contains unresolved references."
)

print("=" * 80)

# Incremental scd2 checks bronze

In [0]:
for resource in FHIR_RESOURCES:

    table_name = (
        f"{CATALOG}.{BRONZE_SCHEMA}.{resource.lower()}"
    )

    print(f"\n===== {table_name} =====")
    spark.table(table_name).printSchema()

In [0]:
# ============================================================
# BRONZE DATA QUALITY VALIDATION
# ============================================================

from pyspark.sql.functions import col

CATALOG = "fhir_assignment"
BRONZE_SCHEMA = "bronze"

BRONZE_TABLES = {
    "Patient": f"{CATALOG}.{BRONZE_SCHEMA}.patient",
    "Encounter": f"{CATALOG}.{BRONZE_SCHEMA}.encounter",
    "Observation": f"{CATALOG}.{BRONZE_SCHEMA}.observation",
    "Condition": f"{CATALOG}.{BRONZE_SCHEMA}.condition"
}

BRONZE_KEYS = {
    "Patient": "resource_id",
    "Encounter": "resource_id",
    "Observation": "resource_id",
    "Condition": "resource_id"
}


print("=" * 80)
print("BRONZE DATA QUALITY VALIDATION")
print("=" * 80)


# ============================================================
# RECORD COUNTS
# ============================================================

print("\nRECORD COUNTS")
print("-" * 80)

for resource, table_name in BRONZE_TABLES.items():

    df = spark.table(table_name)
    count = df.count()

    print(f"{resource:<15} count={count}")


# ============================================================
# PRIMARY KEY NULL CHECK
# ============================================================

print("\nPRIMARY KEY NULL CHECKS")
print("-" * 80)

for resource, table_name in BRONZE_TABLES.items():

    key = BRONZE_KEYS[resource]

    null_count = (
        spark.table(table_name)
        .filter(
            col(key).isNull() |
            (col(key) == "")
        )
        .count()
    )

    status = "PASS" if null_count == 0 else "FAIL"

    print(
        f"{resource:<15}"
        f"{key:<20}"
        f"Null/empty={null_count:<8}"
        f"{status}"
    )


# ============================================================
# DUPLICATE RESOURCE ID CHECK
# ============================================================

print("\nDUPLICATE RESOURCE ID CHECKS")
print("-" * 80)

for resource, table_name in BRONZE_TABLES.items():

    key = BRONZE_KEYS[resource]

    duplicate_count = (
        spark.table(table_name)
        .groupBy(key)
        .count()
        .filter(col("count") > 1)
        .count()
    )

    status = "PASS" if duplicate_count == 0 else "FAIL"

    print(
        f"{resource:<15}"
        f"Duplicate {key}={duplicate_count:<8}"
        f"{status}"
    )


# ============================================================
# REQUIRED COLUMN NULL CHECKS
# ============================================================

print("\nREQUIRED COLUMN NULL CHECKS")
print("-" * 80)

for resource, table_name in BRONZE_TABLES.items():

    df = spark.table(table_name)

    resource_id_nulls = (
        df.filter(col("resource_id").isNull()).count()
    )

    resource_json_nulls = (
        df.filter(col("resource_json").isNull()).count()
    )

    source_file_nulls = (
        df.filter(col("source_file").isNull()).count()
    )

    print(f"\n{resource}")
    print(f"  resource_id null    = {resource_id_nulls}")
    print(f"  resource_json null  = {resource_json_nulls}")
    print(f"  source_file null     = {source_file_nulls}")


# ============================================================
# RECORD HASH / RUN ID CHECK
# ============================================================

print("\nBRONZE AUDIT COLUMN CHECK")
print("-" * 80)

for resource, table_name in BRONZE_TABLES.items():

    df = spark.table(table_name)

    hash_nulls = (
        df.filter(col("record_hash").isNull()).count()
    )

    run_id_nulls = (
        df.filter(col("run_id").isNull()).count()
    )

    print(f"\n{resource}")
    print(f"  record_hash null = {hash_nulls}")
    print(f"  run_id null      = {run_id_nulls}")


# ============================================================
# SOURCE FILE PROFILE
# ============================================================

print("\nSOURCE FILE PROFILE")
print("-" * 80)

for resource, table_name in BRONZE_TABLES.items():

    df = spark.table(table_name)

    source_files = (
        df.select("source_file")
        .distinct()
        .count()
    )

    print(
        f"{resource:<15}"
        f"Distinct source files={source_files}"
    )


print("\n" + "=" * 80)
print("BRONZE DATA QUALITY VALIDATION COMPLETE")
print("=" * 80)

# silver scd2 implementation

1. Initialisation
2. Schema verification
3. Silver validation

In [0]:


 #      ONE-TIME SILVER TABLE INITIALIZATION


silver_tables = {
    "patient": patient_silver,
    "encounter": encounter_silver,
    "observation": observation_silver,
    "condition": condition_silver
}

for table_name, silver_df in silver_tables.items():

    full_table_name = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

    (
        silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

    print(f"Initialized Silver table: {full_table_name}")

In [0]:

 
#  VERIFY SILVER TABLE SCHEMAS

CATALOG = "fhir_assignment"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
SILVER_TABLES = {
    "patient": f"{CATALOG}.{SILVER_SCHEMA}.patient",
    "encounter": f"{CATALOG}.{SILVER_SCHEMA}.encounter",
    "observation": f"{CATALOG}.{SILVER_SCHEMA}.observation",
    "condition": f"{CATALOG}.{SILVER_SCHEMA}.condition"
}

for table_name, full_table_name in SILVER_TABLES.items():

    print(f"\n===== {full_table_name} =====")
    spark.table(full_table_name).printSchema()

In [0]:
from pyspark.sql.functions import col

silver_keys = {
    "patient": "patient_id",
    "encounter": "encounter_id",
    "observation": "observation_id",
    "condition": "condition_id"
}

for table_name, key in silver_keys.items():

    full_table_name = (
        f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    )

    df = spark.table(full_table_name)

    total = df.count()

    distinct_ids = (
        df.select(key)
          .distinct()
          .count()
    )

    duplicate_ids = (
        df.groupBy(key)
          .count()
          .filter(col("count") > 1)
          .count()
    )

    print(
        f"{table_name:<15}"
        f"count={total:<6}"
        f"distinct_ids={distinct_ids:<6}"
        f"duplicate_ids={duplicate_ids}"
    )

In [0]:
# ============================================================
# SILVER DATA QUALITY VALIDATION
# ============================================================

from pyspark.sql.functions import col

CATALOG = "fhir_assignment"
SILVER_SCHEMA = "silver"

SILVER_TABLES = {
    "Patient": f"{CATALOG}.{SILVER_SCHEMA}.patient",
    "Encounter": f"{CATALOG}.{SILVER_SCHEMA}.encounter",
    "Observation": f"{CATALOG}.{SILVER_SCHEMA}.observation",
    "Condition": f"{CATALOG}.{SILVER_SCHEMA}.condition"
}

SILVER_KEYS = {
    "Patient": "patient_id",
    "Encounter": "encounter_id",
    "Observation": "observation_id",
    "Condition": "condition_id"
}


print("=" * 80)
print("SILVER DATA QUALITY VALIDATION")
print("=" * 80)


# ============================================================
# RECORD COUNTS
# ============================================================

print("\nRECORD COUNTS")
print("-" * 80)

for resource, table_name in SILVER_TABLES.items():

    count = spark.table(table_name).count()

    print(
        f"{resource:<15}"
        f"count={count}"
    )


# ============================================================
# PRIMARY KEY NULL CHECK
# ============================================================

print("\nPRIMARY KEY NULL CHECKS")
print("-" * 80)

for resource, table_name in SILVER_TABLES.items():

    key = SILVER_KEYS[resource]

    null_count = (
        spark.table(table_name)
        .filter(
            col(key).isNull() |
            (col(key) == "")
        )
        .count()
    )

    status = "PASS" if null_count == 0 else "FAIL"

    print(
        f"{resource:<15}"
        f"{key:<20}"
        f"Null/empty={null_count:<8}"
        f"{status}"
    )


# ============================================================
# DUPLICATE PRIMARY KEY CHECK
# ============================================================

print("\nDUPLICATE PRIMARY KEY CHECKS")
print("-" * 80)

for resource, table_name in SILVER_TABLES.items():

    key = SILVER_KEYS[resource]

    duplicate_count = (
        spark.table(table_name)
        .groupBy(key)
        .count()
        .filter(col("count") > 1)
        .count()
    )

    status = "PASS" if duplicate_count == 0 else "FAIL"

    print(
        f"{resource:<15}"
        f"Duplicate {key}={duplicate_count:<8}"
        f"{status}"
    )


# ============================================================
# REQUIRED COLUMN NULL CHECKS
# ============================================================

print("\nREQUIRED COLUMN NULL CHECKS")
print("-" * 80)

required_columns = {
    "Patient": [
        "patient_id"
    ],
    "Encounter": [
        "encounter_id"
    ],
    "Observation": [
        "observation_id"
    ],
    "Condition": [
        "condition_id"
    ]
}

for resource, table_name in SILVER_TABLES.items():

    df = spark.table(table_name)

    print(f"\n{resource}")

    for column_name in required_columns[resource]:

        null_count = (
            df.filter(col(column_name).isNull())
            .count()
        )

        print(
            f"  {column_name:<20}"
            f"null={null_count}"
        )


# ============================================================
# AUDIT COLUMN CHECK
# ============================================================

print("\nSILVER AUDIT COLUMN CHECK")
print("-" * 80)

for resource, table_name in SILVER_TABLES.items():

    df = spark.table(table_name)

    record_hash_nulls = (
        df.filter(col("record_hash").isNull())
        .count()
    )

    run_id_nulls = (
        df.filter(col("run_id").isNull())
        .count()
    )

    print(f"\n{resource}")
    print(f"  record_hash null = {record_hash_nulls}")
    print(f"  run_id null      = {run_id_nulls}")


# ============================================================
# DATA TYPE / BUSINESS FIELD PROFILE
# ============================================================

print("\nBUSINESS FIELD NULL PROFILE")
print("-" * 80)

business_columns = {
    "Patient": [
        "gender",
        "birth_date",
        "identifier_value",
        "city",
        "state",
        "postal_code"
    ],

    "Encounter": [
        "status",
        "class_code",
        "period_start",
        "period_end",
        "patient_id"
    ],

    "Observation": [
        "status",
        "category_code",
        "observation_code",
        "effective_datetime",
        "patient_id"
    ],

    "Condition": [
        "clinical_status",
        "condition_code",
        "condition_display",
        "patient_id"
    ]
}

for resource, table_name in SILVER_TABLES.items():

    df = spark.table(table_name)

    print(f"\n{resource}")

    for column_name in business_columns[resource]:

        null_count = (
            df.filter(col(column_name).isNull())
            .count()
        )

        populated_count = df.count() - null_count

        print(
            f"  {column_name:<25}"
            f"Populated={populated_count:<6}"
            f"Null={null_count}"
        )


# ============================================================
# REFERENTIAL RELATIONSHIP PROFILE
# ============================================================

print("\nSILVER RELATIONSHIP PROFILE")
print("-" * 80)

patient_ids = (
    spark.table(SILVER_TABLES["Patient"])
    .select("patient_id")
    .distinct()
)

encounter_ids = (
    spark.table(SILVER_TABLES["Encounter"])
    .select("encounter_id")
    .distinct()
)


encounter_patient_unresolved = (
    spark.table(SILVER_TABLES["Encounter"])
    .filter(col("patient_id").isNotNull())
    .join(
        patient_ids,
        "patient_id",
        "left_anti"
    )
    .count()
)


observation_patient_unresolved = (
    spark.table(SILVER_TABLES["Observation"])
    .filter(col("patient_id").isNotNull())
    .join(
        patient_ids,
        "patient_id",
        "left_anti"
    )
    .count()
)


condition_patient_unresolved = (
    spark.table(SILVER_TABLES["Condition"])
    .filter(col("patient_id").isNotNull())
    .join(
        patient_ids,
        "patient_id",
        "left_anti"
    )
    .count()
)


condition_encounter_unresolved = (
    spark.table(SILVER_TABLES["Condition"])
    .filter(col("encounter_id").isNotNull())
    .join(
        encounter_ids,
        "encounter_id",
        "left_anti"
    )
    .count()
)


print(
    f"Encounter -> Patient unresolved : "
    f"{encounter_patient_unresolved}"
)

print(
    f"Observation -> Patient unresolved : "
    f"{observation_patient_unresolved}"
)

print(
    f"Condition -> Patient unresolved : "
    f"{condition_patient_unresolved}"
)

print(
    f"Condition -> Encounter unresolved : "
    f"{condition_encounter_unresolved}"
)


print("\n" + "=" * 80)
print("SILVER DATA QUALITY VALIDATION COMPLETE")
print("=" * 80)

# Gold validation and testing after SCD2

In [0]:
# ============================================================
# VERIFY GOLD TABLE SCHEMAS
# ============================================================
GOLD_SCHEMA = "gold"
GOLD_TABLES = {
    "patient": f"{CATALOG}.{GOLD_SCHEMA}.dim_patient",
    "encounter": f"{CATALOG}.{GOLD_SCHEMA}.fact_encounter",
    "observation": f"{CATALOG}.{GOLD_SCHEMA}.fact_observation",
    "condition": f"{CATALOG}.{GOLD_SCHEMA}.fact_condition"
}

for table_name, full_table_name in GOLD_TABLES.items():

    print(f"\n===== {full_table_name} =====")
    spark.table(full_table_name).printSchema()

In [0]:
# ============================================================
# GOLD DATA QUALITY VALIDATION
# ============================================================

from pyspark.sql.functions import col

CATALOG = "fhir_assignment"
GOLD_SCHEMA = "gold"

GOLD_TABLES = {
    "Patient": f"{CATALOG}.{GOLD_SCHEMA}.dim_patient",
    "Encounter": f"{CATALOG}.{GOLD_SCHEMA}.fact_encounter",
    "Observation": f"{CATALOG}.{GOLD_SCHEMA}.fact_observation",
    "Condition": f"{CATALOG}.{GOLD_SCHEMA}.fact_condition"
}

GOLD_KEYS = {
    "Patient": "patient_id",
    "Encounter": "encounter_id",
    "Observation": "observation_id",
    "Condition": "condition_id"
}

EXPECTED_COUNTS = {
    "Patient": 261,
    "Encounter": 92,
    "Observation": 4578,
    "Condition": 48
}


print("=" * 80)
print("GOLD DATA QUALITY VALIDATION")
print("=" * 80)


# ============================================================
# RECORD COUNTS
# ============================================================

print("\nRECORD COUNTS")
print("-" * 80)

silver_tables = {
    "Patient": f"{CATALOG}.silver.patient",
    "Encounter": f"{CATALOG}.silver.encounter",
    "Observation": f"{CATALOG}.silver.observation",
    "Condition": f"{CATALOG}.silver.condition"
}

gold_tables = {
    "Patient": f"{CATALOG}.gold.dim_patient",
    "Encounter": f"{CATALOG}.gold.fact_encounter",
    "Observation": f"{CATALOG}.gold.fact_observation",
    "Condition": f"{CATALOG}.gold.fact_condition"
}

for resource in gold_tables:

    actual = spark.table(
        gold_tables[resource]
    ).count()

    expected = spark.table(
        silver_tables[resource]
    ).count()

    status = "PASS" if actual == expected else "FAIL"

    print(
        f"{resource:<15}"
        f"Actual={actual:<8}"
        f"Expected Silver={expected:<8}"
        f"{status}"
    )

# ============================================================
# PRIMARY KEY NULL CHECKS
# ============================================================

print("\nPRIMARY KEY NULL CHECKS")
print("-" * 80)

for resource, table_name in GOLD_TABLES.items():

    key = GOLD_KEYS[resource]

    null_count = (
        spark.table(table_name)
        .filter(
            col(key).isNull() |
            (col(key) == "")
        )
        .count()
    )

    status = "PASS" if null_count == 0 else "FAIL"

    print(
        f"{resource:<15}"
        f"{key:<20}"
        f"Null/empty={null_count:<8}"
        f"{status}"
    )


# ============================================================
# DUPLICATE PRIMARY KEY CHECKS
# ============================================================

print("\nDUPLICATE PRIMARY KEY CHECKS")
print("-" * 80)

for resource, table_name in GOLD_TABLES.items():

    key = GOLD_KEYS[resource]

    duplicate_count = (
        spark.table(table_name)
        .groupBy(key)
        .count()
        .filter(col("count") > 1)
        .count()
    )

    status = "PASS" if duplicate_count == 0 else "FAIL"

    print(
        f"{resource:<15}"
        f"Duplicate {key}={duplicate_count:<8}"
        f"{status}"
    )


# ============================================================
# DATE / TIMESTAMP PROFILE
# ============================================================

print("\nDATE / TIMESTAMP PROFILE")
print("-" * 80)

patient_df = spark.table(GOLD_TABLES["Patient"])
encounter_df = spark.table(GOLD_TABLES["Encounter"])
observation_df = spark.table(GOLD_TABLES["Observation"])
condition_df = spark.table(GOLD_TABLES["Condition"])

date_profiles = [
    ("Patient.birth_date", patient_df, "birth_date"),
    ("Encounter.period_start", encounter_df, "period_start"),
    ("Encounter.period_end", encounter_df, "period_end"),
    ("Observation.effective_datetime", observation_df, "effective_datetime"),
    ("Condition.onset_date", condition_df, "onset_date")
]

for label, df, column_name in date_profiles:

    populated = (
        df.filter(col(column_name).isNotNull())
        .count()
    )

    total = df.count()
    null_count = total - populated

    print(
        f"{label:<40}"
        f"Populated={populated:<8}"
        f"Null={null_count}"
    )


# ============================================================
# ENCOUNTER DURATION VALIDATION
# ============================================================

print("\nENCOUNTER DURATION VALIDATION")
print("-" * 80)

negative_duration = (
    encounter_df
    .filter(col("duration_hours") < 0)
    .count()
)

print(f"Negative duration records: {negative_duration}")

if negative_duration == 0:
    print("PASS: No negative encounter durations")
else:
    print("FAIL: Negative encounter durations found")


# ============================================================
# OBSERVATION VALUE TYPE VALIDATION
# ============================================================

print("\nOBSERVATION VALUE TYPE VALIDATION")
print("-" * 80)

(
    observation_df
    .groupBy("value_type")
    .count()
    .orderBy("value_type")
    .show()
)


# ============================================================
# GOLD RELATIONSHIP PROFILE
# ============================================================

# ============================================================
# GOLD RELATIONSHIP PROFILE
# ============================================================

print("\nGOLD RELATIONSHIP PROFILE")
print("-" * 80)

patient_ids = (
    patient_df
    .select("patient_id")
    .distinct()
)

encounter_ids = (
    encounter_df
    .select("encounter_id")
    .distinct()
)

# Encounter -> Patient
encounter_unresolved = (
    encounter_df
    .filter(col("patient_id").isNotNull())
    .join(
        patient_ids,
        encounter_df.patient_id == patient_ids.patient_id,
        "left_anti"
    )
    .count()
)

# Observation -> Patient
observation_unresolved = (
    observation_df
    .filter(col("patient_id").isNotNull())
    .join(
        patient_ids,
        observation_df.patient_id == patient_ids.patient_id,
        "left_anti"
    )
    .count()
)

# Condition -> Patient
condition_patient_unresolved = (
    condition_df
    .filter(col("patient_id").isNotNull())
    .join(
        patient_ids,
        condition_df.patient_id == patient_ids.patient_id,
        "left_anti"
    )
    .count()
)

# Condition -> Encounter
condition_encounter_unresolved = (
    condition_df
    .filter(col("encounter_id").isNotNull())
    .join(
        encounter_ids,
        condition_df.encounter_id == encounter_ids.encounter_id,
        "left_anti"
    )
    .count()
)

print(f"Encounter -> Patient : {encounter_unresolved}")
print(f"Observation -> Patient : {observation_unresolved}")
print(f"Condition -> Patient : {condition_patient_unresolved}")
print(f"Condition -> Encounter : {condition_encounter_unresolved}")


print("\n" + "=" * 80)
print("GOLD DATA QUALITY SUMMARY")
print("=" * 80)

print("PASS: Record counts match expected Silver counts")
print("PASS: Primary keys contain no null/empty values")
print("PASS: Primary keys contain no duplicates")
print("PASS: No negative encounter durations")
print("PASS: Gold transformations completed successfully")

print(
    "\nReferential integrity is reported separately because "
    "the source FHIR dataset contains unresolved references."
)

print("=" * 80)

In [0]:
# ============================================================
# GOLD RELATIONSHIP PROFILE
# ============================================================

print("\nGOLD RELATIONSHIP PROFILE")
print("-" * 80)

patient_ids = (
    patient_df
    .select("patient_id")
    .distinct()
)

encounter_ids = (
    encounter_df
    .select("encounter_id")
    .distinct()
)

# Encounter -> Patient
# Only non-null patient IDs are checked.
encounter_unresolved = (
    encounter_df
    .filter(col("patient_id").isNotNull())
    .join(
        patient_ids,
        encounter_df.patient_id == patient_ids.patient_id,
        "left_anti"
    )
    .count()
)

# Observation -> Patient
# Only non-null patient IDs are checked.
observation_unresolved = (
    observation_df
    .filter(col("patient_id").isNotNull())
    .join(
        patient_ids,
        observation_df.patient_id == patient_ids.patient_id,
        "left_anti"
    )
    .count()
)

# Condition -> Patient
# Only non-null patient IDs are checked.
condition_patient_unresolved = (
    condition_df
    .filter(col("patient_id").isNotNull())
    .join(
        patient_ids,
        condition_df.patient_id == patient_ids.patient_id,
        "left_anti"
    )
    .count()
)

# Condition -> Encounter
# Only non-null encounter IDs are checked.
condition_encounter_unresolved = (
    condition_df
    .filter(col("encounter_id").isNotNull())
    .join(
        encounter_ids,
        condition_df.encounter_id == encounter_ids.encounter_id,
        "left_anti"
    )
    .count()
)

print(f"Encounter -> Patient : {encounter_unresolved}")
print(f"Observation -> Patient : {observation_unresolved}")
print(f"Condition -> Patient : {condition_patient_unresolved}")
print(f"Condition -> Encounter : {condition_encounter_unresolved}")

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT patient_id) AS distinct_patient_ids,
    COUNT(*) - COUNT(DISTINCT patient_id) AS difference
FROM fhir_assignment.gold.dim_patient;

In [0]:
%sql
SELECT patient_id, COUNT(*) AS cnt
FROM fhir_assignment.gold.dim_patient
GROUP BY patient_id
HAVING COUNT(*) > 1
ORDER BY cnt DESC;